## 2.1 Recolección de datos

Los datos de PEMS-04 provienen de la red de sensores de tráfico **Caltrans
PeMS** (Performance Measurement System) del distrito 4 (área de la Bahía de
San Francisco), agregados en intervalos de 5 minutos. El conjunto fue
curado y publicado por el grupo de LSTTN (Luo et al., 2024) junto con los
otros tres datasets de la Tabla 1 del Entregable 1, disponible en el Google
Drive enlazado en su repositorio (https://github.com/GeoX-Lab/LSTTN), en el
mismo formato estandarizado que usan trabajos previos como ASTGCN y STSGCN.

Se descargaron dos archivos:
- `PEMS04.npz`: tensor crudo de tráfico, forma `(T, N, C)` = `(16992, 307, 3)`.
- `distance.csv`: lista de aristas `from, to, distance` que describe la
  conectividad vial física entre sensores (distancia en metros/km).

**Limitaciones encontradas:** el dataset no incluye coordenadas geográficas
de los sensores (solo IDs y distancias relativas), por lo que la
visualización del grafo se hace con un layout basado en la topología
(spring layout) en vez de un mapa real. Tampoco se distribuye una matriz de
adyacencia ya construida: debe derivarse de `distance.csv` (sección 2.4).


In [ ]:
# Dependencias
!pip install numpy pandas networkx matplotlib torch torch-geometric

In [ ]:
# Paths (ajustar a la ubicación local de los archivos)
DATA_DIR = "raw_data/PEMS04"
NPZ_PATH = f"{DATA_DIR}/PEMS04.npz"
DIST_PATH = f"{DATA_DIR}/PEMS04.csv"

import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import torch
from torch_geometric.data import Data

np.random.seed(42)


In [ ]:
# Carga de datos crudos
raw = np.load(NPZ_PATH)["data"]                 # shape (T, N, C)
distance_df = pd.read_csv(DIST_PATH)             # columns: from, to, distance

T, N, C = raw.shape
FEATURE_NAMES = ["flujo", "ocupacion", "velocidad"]
print(f"Tensor crudo: T={T} pasos, N={N} sensores, C={C} caracteristicas")
distance_df.head()


## 2.2 Descripción del dataset

A continuación se describe el dataset en su estado crudo: dimensiones,
tipos de dato, rango de valores por característica y proporción de
valores faltantes. Esto justifica las decisiones de preprocesamiento de la
sección 2.3.


In [ ]:
# Estadisticas descriptivas por canal
for c in range(C):
    ch = raw[:, :, c]
    print(f"{FEATURE_NAMES[c]:>10} | min={ch.min():8.2f}  max={ch.max():8.2f}  "
          f"media={ch.mean():8.2f}  std={ch.std():8.2f}")

# Valores faltantes: en PEMS04 los sensores caidos quedan registrados como 0
# en el canal de flujo. Se cuantifica esa proporcion por canal.
for c in range(C):
    zero_ratio = (raw[:, :, c] == 0).mean()
    print(f"Proporcion de ceros en {FEATURE_NAMES[c]}: {zero_ratio:.4%}")


In [ ]:
# Visualizacion de una serie cruda (un sensor, una semana) para inspeccion
sensor_id = 0
window = slice(0, 7 * 24 * 12)  # 7 dias a 5 min/paso

fig, axes = plt.subplots(C, 1, figsize=(10, 6), sharex=True)
for c in range(C):
    axes[c].plot(raw[window, sensor_id, c])
    axes[c].set_ylabel(FEATURE_NAMES[c])
axes[-1].set_xlabel("paso temporal (5 min)")
fig.suptitle(f"Serie cruda - sensor {sensor_id} (primera semana)")
plt.tight_layout()
plt.show()


## 2.3 Preprocesamiento de datos

Pasos aplicados, en orden, antes de construir el grafo y las ventanas de
entrenamiento:

1. **Manejo de valores faltantes**: los ceros del canal de flujo se tratan
   como lecturas faltantes y se interpolan linealmente en el eje temporal,
   por sensor y por canal.
2. **Particionamiento temporal** train/val/test (60/20/20), secuencial
   (sin mezclar el orden temporal, para evitar fuga de información futura).
3. **Normalización** z-score por canal, con media y desviación calculadas
   **solo sobre el split de entrenamiento** y reutilizadas en val/test.
4. **Ventaneo (sliding window)** para generar las secuencias de entrada/
   salida que usará el modelo en el Entregable 3 (12 pasos de historia →
   12 pasos de pronóstico, equivalente a 1 hora de contexto y 1 hora de
   horizonte).


In [ ]:
def interpolate_missing(arr, missing_channel=0):
    """Linear interpolation along the time axis for zero-valued (missing) readings."""
    arr = arr.copy()
    for n in range(arr.shape[1]):
        series = arr[:, n, missing_channel]
        miss = series == 0
        if miss.any() and not miss.all():
            idx = np.arange(len(series))
            series[miss] = np.interp(idx[miss], idx[~miss], series[~miss])
            arr[:, n, missing_channel] = series
    return arr

clean = interpolate_missing(raw, missing_channel=0)
print("Ceros restantes en flujo tras interpolar:", (clean[:, :, 0] == 0).mean())


In [ ]:
# Particion temporal secuencial 60/20/20
n_train = int(T * 0.6)
n_val = int(T * 0.2)

train_raw = clean[:n_train]
val_raw = clean[n_train:n_train + n_val]
test_raw = clean[n_train + n_val:]
print("train/val/test shapes:", train_raw.shape, val_raw.shape, test_raw.shape)


In [ ]:
# Normalizacion z-score (estadisticas calculadas solo con train)
mean = train_raw.mean(axis=(0, 1), keepdims=True)
std = train_raw.std(axis=(0, 1), keepdims=True) + 1e-8

train_norm = (train_raw - mean) / std
val_norm = (val_raw - mean) / std
test_norm = (test_raw - mean) / std

print("media post-norm (train):", train_norm.mean().round(4))
print("std post-norm (train):", train_norm.std().round(4))


In [ ]:
def sliding_window(arr, in_len=12, out_len=12):
    """Build (X, Y) supervised pairs via a sliding window over the time axis."""
    n_steps = arr.shape[0] - in_len - out_len + 1
    X = np.stack([arr[t:t + in_len] for t in range(n_steps)])
    Y = np.stack([arr[t + in_len:t + in_len + out_len] for t in range(n_steps)])
    return X, Y

X_train, Y_train = sliding_window(train_norm)
X_val, Y_val = sliding_window(val_norm)
X_test, Y_test = sliding_window(test_norm)

print("X_train:", X_train.shape, "| Y_train:", Y_train.shape)


## 2.4 Construcción del grafo

Siguiendo el modelamiento del Entregable 1 (grafo $G=(V,E,A)$ dirigido y
ponderado):

- **Nodos ($V$)**: un nodo por sensor (`N = 307`). El atributo de cada nodo
  es el vector de características promedio en el split de entrenamiento
  (flujo, ocupación, velocidad), usado solo para inspección/visualización
  estática; el tensor temporal completo (`X_train`, `X_val`, `X_test`) es el
  que alimentará al modelo en el Entregable 3.
- **Aristas ($E$)**: se toman directamente de `distance.csv`. Cada fila
  `(from, to, distance)` define una arista dirigida que respeta el sentido
  real de circulación vial.
- **Matriz de adyacencia ($A$)**: se pondera cada arista con un kernel
  gaussiano sobre la distancia, $w_{ij} = \exp(-d_{ij}^2/\sigma^2)$, donde
  $\sigma$ es la desviación estándar de las distancias observadas. Esto da
  más peso a sensores físicamente cercanos.


In [ ]:
def build_adjacency_matrix(distance_df, num_nodes):
    """Bidirectional binary connectivity (matches LSTTN's get_adjacency_matrix_2direction);
    a separate gaussian-weighted matrix is kept for edge_attr."""
    A_bin = np.zeros((num_nodes, num_nodes), dtype=np.float32)
    A_weighted = np.zeros((num_nodes, num_nodes), dtype=np.float32)
    dists = distance_df["cost"].values
    sigma2 = dists.std() ** 2
    for f, t, d in zip(distance_df["from"], distance_df["to"], distance_df["cost"]):
        f, t = int(f), int(t)
        w = np.exp(-(d ** 2) / sigma2)
        A_bin[f, t] = 1
        A_bin[t, f] = 1
        A_weighted[f, t] = w
        A_weighted[t, f] = w
    return A_bin, A_weighted

A_bin, A = build_adjacency_matrix(distance_df, N)
n_edges_no_loops = int((A_bin > 0).sum())
print("Aristas sin self-loops:", n_edges_no_loops)
print("Aristas + self-loops (convencion GCN, A+I):", n_edges_no_loops + N)
print("Densidad:", n_edges_no_loops / (N * N))

In [ ]:
# Objeto Data de PyTorch Geometric
rows, cols = np.nonzero(A_bin)
edge_index = torch.tensor(np.array([rows, cols]), dtype=torch.long)
edge_weight = torch.tensor(A[rows, cols], dtype=torch.float)

node_features = torch.tensor(train_raw.mean(axis=0), dtype=torch.float)  # (N, C)
graph_data = Data(x=node_features, edge_index=edge_index, edge_attr=edge_weight)
print(graph_data)


In [ ]:
# Visualizacion del grafo (layout topologico: no hay coordenadas geograficas)
G = nx.Graph()
G.add_nodes_from(range(N))
rows, cols = np.nonzero(A_bin)
G.add_edges_from((int(r), int(c), {"weight": A[r, c]}) for r, c in zip(rows, cols) if r <= c)

pos = nx.spring_layout(G, seed=42, k=0.3)
weights = [G[u][v]["weight"] * 4 for u, v in G.edges()]

plt.figure(figsize=(9, 9))
nx.draw_networkx_nodes(G, pos, node_size=25, node_color="#3b82f6")
nx.draw_networkx_edges(G, pos, width=weights, alpha=0.3)
plt.title("Grafo de sensores PEMS-04 (no dirigido, ponderado)")
plt.axis("off")
plt.show()


## 2.5 Análisis estructural del grafo

Métricas básicas a nivel de nodo, arista y grafo completo, sobre el grafo
dirigido construido en 2.4.


In [ ]:
deg = dict(G.degree())

print(f"Nodos: {G.number_of_nodes()} | Aristas: {G.number_of_edges()}")
print(f"Densidad: {nx.density(G):.5f}")
print(f"Grado promedio: {np.mean(list(deg.values())):.2f}")
print(f"Componentes conexas: {nx.number_connected_components(G)}")
print(f"Coeficiente de clustering promedio: {nx.average_clustering(G):.4f}")

In [ ]:
plt.figure(figsize=(6, 4))
plt.hist(list(deg.values()), bins=20, color="#3b82f6")
plt.title("Distribución de grado")
plt.xlabel("grado")
plt.ylabel("frecuencia")
plt.tight_layout()
plt.show()